In [18]:
!pip install -q google-generativeai transformers torch
print("Dependencies installed.")

Dependencies installed.


In [39]:
from openai import OpenAI
from google.colab import userdata
import json
import time

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

# Use OpenRouter's free auto-router — picks any available free model dynamically
MODEL_NAME = "openrouter/auto"

print(f"OpenRouter configured successfully with model: {MODEL_NAME}")

OpenRouter configured successfully with model: openrouter/auto


In [35]:
REVISED_PROMPT = """You are an ESG operations triage classifier for a large Australian organisation.
Your role is to extract structured fields from an inbound operational message so it can be routed automatically.

## Controlled vocabularies (you MUST use one of these exact values)

issue_category: ENERGY_WASTE | WATER_LEAK | WASTE_CONTAMINATION | ACCESSIBILITY_BARRIER | SUSPECTED_SUPPLIER_NONCOMPLIANCE | OTHER

urgency:
- LOW       = inefficiency, no immediate harm or financial impact (e.g. one light left on)
- MEDIUM    = ongoing waste or minor accessibility issue, action needed within 24h
- HIGH      = active leak/damage, accessibility barrier blocking access now, action needed within 4h
- CRITICAL  = safety risk, environmental hazard, or potential regulatory breach, action needed immediately

sentiment: POSITIVE | NEUTRAL | NEGATIVE

followup_required: Y if the reporter needs an update or the issue requires further investigation; N otherwise.

recommended_team: FACILITIES | SUSTAINABILITY | ACCESSIBILITY_TEAM | PROCUREMENT | SECURITY | NONE

data_sensitivity_risk: LOW | MEDIUM | HIGH
- LOW    = operational facts only, no personal data
- MEDIUM = location + reporter identifiable indirectly
- HIGH   = identifies named individuals, suppliers under investigation, or whistleblowing-adjacent content

## Output rules
- Return ONLY a single valid JSON object. No prose, no markdown fences.
- If the message is empty, non-ESG-related, or unintelligible, set issue_category="OTHER", recommended_team="NONE", and put your reason in escalation_reason.
- brief_summary: max 20 words, factual, no opinion.
- escalation_reason: one sentence explaining why this should or should not be escalated.

## Example
Input: "There is a water leak in Building C that has been running all morning."
Output:
{"issue_category":"WATER_LEAK","urgency":"HIGH","sentiment":"NEGATIVE","followup_required":"Y","recommended_team":"FACILITIES","escalation_reason":"Active leak with ongoing resource waste, requires facilities response within 4 hours.","data_sensitivity_risk":"LOW","brief_summary":"Water leak in Building C running since this morning."}

## Now classify this message
Message: {MESSAGE}
"""

print("Revised prompt template defined.")
print(f"Length: {len(REVISED_PROMPT)} characters")

Revised prompt template defined.
Length: 2173 characters


In [36]:
test_messages = [
    "There is a water leak in Building C that has been running all morning.",
    "The recycling bins are contaminated again and no one seems to be checking them.",
    "I want to report that one of our suppliers may not meet our sustainability policy. They are using uncertified palm oil and I have evidence from the loading dock."
]

for i, msg in enumerate(test_messages, 1):
    print(f"Message {i}: {msg}\n")

Message 1: There is a water leak in Building C that has been running all morning.

Message 2: The recycling bins are contaminated again and no one seems to be checking them.

Message 3: I want to report that one of our suppliers may not meet our sustainability policy. They are using uncertified palm oil and I have evidence from the loading dock.



In [40]:
gemini_results = []

def call_llm_with_retry(prompt, max_retries=3):
    """Call OpenRouter with automatic retry on 429 rate limits."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": "You are an ESG operations triage classifier. Return only valid JSON with no markdown fences or extra text."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                extra_headers={
                    "HTTP-Referer": "https://github.com",
                    "X-Title": "BUS5001 Q3 ESG Triage"
                }
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            error_str = str(e)
            if "429" in error_str and attempt < max_retries - 1:
                wait = 35  # wait slightly longer than the 29s server hint
                print(f"  Rate limited, waiting {wait}s (attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait)
                continue
            raise

for i, msg in enumerate(test_messages, 1):
    prompt = REVISED_PROMPT.replace("{MESSAGE}", msg)
    try:
        raw_text = call_llm_with_retry(prompt)
        # Strip markdown fences if model added them
        if raw_text.startswith("```"):
            raw_text = raw_text.split("```")[1]
            if raw_text.startswith("json"):
                raw_text = raw_text[4:]
            raw_text = raw_text.strip()
        parsed = json.loads(raw_text)
        gemini_results.append({"message": msg, "result": parsed})
        print(f"--- Message {i} ---")
        print(json.dumps(parsed, indent=2))
        print()
        time.sleep(5)  # bigger gap between messages to avoid hitting limits
    except Exception as e:
        print(f"Error on message {i}: {e}")
        gemini_results.append({"message": msg, "result": None, "error": str(e)})

--- Message 1 ---
{
  "issue_category": "WATER_LEAK",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "FACILITIES",
  "escalation_reason": "Active leak with ongoing resource waste, requires facilities response within 4 hours.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "Water leak in Building C running since this morning."
}

--- Message 2 ---
{
  "issue_category": "WASTE_CONTAMINATION",
  "urgency": "MEDIUM",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "SUSTAINABILITY",
  "escalation_reason": "Recycling bins are contaminated, indicating a potential ongoing issue with waste management practices that requires investigation by the sustainability team.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "Recycling bins are contaminated, and there is a perceived lack of oversight."
}

--- Message 3 ---
{
  "issue_category": "SUSPECTED_SUPPLIER_NONCOMPLIANCE",
  "urgency": "CRITICAL",
  "sentime

In [41]:
from transformers import pipeline

# Load the zero-shot classifier (downloads ~1.6GB first time, ~30 seconds on Colab)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Define the same controlled vocabulary as labels
candidate_labels = [
    "energy waste",
    "water leak",
    "waste contamination",
    "accessibility barrier",
    "suspected supplier noncompliance",
    "other"
]

baseline_results = []

for i, msg in enumerate(test_messages, 1):
    result = classifier(msg, candidate_labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    baseline_results.append({
        "message": msg,
        "top_label": top_label,
        "confidence": round(top_score, 3),
        "all_scores": dict(zip(result['labels'], [round(s, 3) for s in result['scores']]))
    })
    print(f"--- Message {i} ---")
    print(f"Top label: {top_label} (confidence {top_score:.3f})")
    print(f"All scores: {baseline_results[-1]['all_scores']}\n")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

--- Message 1 ---
Top label: water leak (confidence 0.950)
All scores: {'water leak': 0.95, 'energy waste': 0.015, 'other': 0.014, 'accessibility barrier': 0.01, 'waste contamination': 0.006, 'suspected supplier noncompliance': 0.005}

--- Message 2 ---
Top label: waste contamination (confidence 0.858)
All scores: {'waste contamination': 0.858, 'other': 0.062, 'accessibility barrier': 0.048, 'suspected supplier noncompliance': 0.025, 'energy waste': 0.006, 'water leak': 0.003}

--- Message 3 ---
Top label: suspected supplier noncompliance (confidence 0.979)
All scores: {'suspected supplier noncompliance': 0.979, 'other': 0.013, 'accessibility barrier': 0.004, 'waste contamination': 0.002, 'energy waste': 0.001, 'water leak': 0.001}



In [42]:
import pandas as pd

comparison = []
for i, msg in enumerate(test_messages):
    gem = gemini_results[i]["result"]
    base = baseline_results[i]
    comparison.append({
        "message_preview": msg[:60] + "...",
        "gemini_category": gem["issue_category"] if gem else "ERROR",
        "gemini_urgency": gem["urgency"] if gem else "ERROR",
        "gemini_team": gem["recommended_team"] if gem else "ERROR",
        "baseline_label": base["top_label"],
        "baseline_confidence": base["confidence"]
    })

df = pd.DataFrame(comparison)
df

,message_preview,gemini_category,gemini_urgency,gemini_team,baseline_label,baseline_confidence
0,There is a water leak in Building C that has b...,WATER_LEAK,HIGH,FACILITIES,water leak,0.950
1,The recycling bins are contaminated again and ...,WASTE_CONTAMINATION,MEDIUM,SUSTAINABILITY,waste contamination,0.858
2,I want to report that one of our suppliers may...,SUSPECTED_SUPPLIER_NONCOMPLIANCE,CRITICAL,PROCUREMENT,suspected supplier noncompliance,0.979


In [45]:
output = {
    "experiment": "BUS5001 Q3 ESG Message Triage",
    "date": "2026-06-12",
    "model_llm": "meta-llama/llama-4-scout:free (via OpenRouter)",
    "model_baseline": "facebook/bart-large-mnli",
    "messages": test_messages,
    "gemini_results": gemini_results,
    "baseline_results": baseline_results
}

with open("experiment_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved experiment_results.json — download it for your GitHub repo.")

Saved experiment_results.json — download it for your GitHub repo.
